# Defend the center: scripted strategy baselines

No training or model checkpoint is required. Each policy directly chooses the tuple
(turn, attack): turn 0=none, 1=left, 2=right; attack 0=off, 1=on.
Compare against the ANN/LIF validation results on the same seeds, frame skip, scenario
and difficulty. These rules are baselines, not instructions given to the learned agents.
Use a fresh Linux kernel with the same Sample Factory/ViZDoom packages. No GPU required.


In [ ]:
import os, sys, json, time, functools
from pathlib import Path
from importlib.metadata import version
import numpy as np
import gymnasium as gym
from sample_factory.cfg.arguments import parse_sf_args, parse_full_cfg
from sample_factory.envs.create_env import create_env
from sample_factory.envs.env_utils import register_env
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "defend the center").is_dir() and (p / "conversion_friendly_v1").is_dir()), None)
if ROOT is None: raise RuntimeError("Start Jupyter in this repository")
OUTPUT_ROOT = Path(os.environ.get("DEFEND_OUTPUT_ROOT", str(ROOT / "defend the center" / "outputs"))).expanduser().resolve()
for spec in DOOM_ENVS:
    register_env(spec.name, functools.partial(make_doom_env_from_spec, spec))


## Settings

Match the learned-model evaluation settings. Keep smoke mode for the first execution, then turn it off for the 300-episode comparison. A report tag prevents overwriting previous results.

In [ ]:
ENV_NAME = "doom_defend_the_center"
SMOKE_TEST = True
N_EPISODES = 10 if SMOKE_TEST else 300
EVAL_SEEDS = list(range(10000, 10000 + N_EPISODES))
ACTION_SEED_OFFSET = 1000000   # independent stream for the random policy
FRAME_SKIP = 4
ALTERNATE_EVERY = 16          # environment decisions, not underlying game tics
REPORT_TAG = "v1_smoke" if SMOKE_TEST else "v1_full"
REPORT_DIR = OUTPUT_ROOT / "strategy_baselines" / REPORT_TAG
if REPORT_DIR.exists():
    raise FileExistsError("Choose a new REPORT_TAG to preserve earlier measurements")
REPORT_DIR.mkdir(parents=True)
argv = [f"--env={ENV_NAME}", "--experiment=scripted_defend_probe",
        f"--train_dir={OUTPUT_ROOT / 'baseline_env_configs'}",
        "--res_h=72", "--res_w=128", f"--env_frameskip={FRAME_SKIP}", "--use_rnn=False"]
parser,_ = parse_sf_args(argv=argv,evaluation=True)
add_doom_env_args(parser); doom_override_defaults(parser)
cfg = parse_full_cfg(parser,argv)
print("Episodes per strategy:",N_EPISODES,"frame skip:",cfg.env_frameskip)


## Scripted action rules

Random samples both factors uniformly. All shooting rules hold attack on. Alternating direction is an additional diagnostic, not a tuned policy. Idle provides a control.

In [ ]:
STRATEGIES = ["random", "idle", "shoot_stationary", "turn_left_shoot",
              "turn_right_shoot", "alternate_turn_shoot"]
def scripted_action(strategy, decision, rng):
    if strategy == "random": return (int(rng.integers(3)), int(rng.integers(2)))
    if strategy == "idle": return (0,0)
    if strategy == "shoot_stationary": return (0,1)
    if strategy == "turn_left_shoot": return (1,1)
    if strategy == "turn_right_shoot": return (2,1)
    if strategy == "alternate_turn_shoot":
        return (1 if (decision // ALTERNATE_EVERY) % 2 == 0 else 2,1)
    raise ValueError(strategy)


## Run matched-seed episodes

A new environment is created for each strategy. Progress is saved after each strategy, so completed results survive a later failure. Episode return is the primary metric; decision count is not a measured kill count or wall-clock survival time.

In [ ]:
all_records = {}; summaries = []
for strategy in STRATEGIES:
    env = create_env(ENV_NAME,cfg=cfg,env_config=None)
    records = []; start = time.time()
    try:
        assert isinstance(env.action_space,gym.spaces.Tuple)
        assert [s.n for s in env.action_space.spaces] == [3,2]
        for ep_seed in EVAL_SEEDS:
            rng = np.random.default_rng(ACTION_SEED_OFFSET + ep_seed)
            obs,_ = env.reset(seed=ep_seed)
            done = truncated = False; total = 0.; decisions = 0
            while not (done or truncated):
                action = scripted_action(strategy,decisions,rng)
                assert env.action_space.contains(action), action
                obs,reward,done,truncated,_ = env.step(action)
                total += float(reward); decisions += 1
            records.append({"seed":ep_seed,"return":total,"decisions":decisions,
                            "terminated":bool(done),"truncated":bool(truncated)})
            if len(records) % 10 == 0: print(strategy,len(records),"/",N_EPISODES)
    finally:
        env.close()
    rewards = np.array([r["return"] for r in records])
    summary = {"strategy":strategy,"episodes":len(records),"mean_return":float(rewards.mean()),
               "std_return":float(rewards.std()),"SE":float(rewards.std()/np.sqrt(len(rewards))),
               "mean_decisions":float(np.mean([r["decisions"] for r in records])),
               "evaluation_seconds":time.time()-start}
    all_records[strategy] = records; summaries.append(summary)
    (REPORT_DIR / f"{strategy}.json").write_text(json.dumps({"summary":summary,"episodes":records},indent=2))
    print(summary)
report = {"env":ENV_NAME,"frame_skip":cfg.env_frameskip,"eval_seeds":EVAL_SEEDS,
          "alternate_every":ALTERNATE_EVERY,"action_seed_offset":ACTION_SEED_OFFSET,
          "packages":{p:version(p) for p in ["sample-factory","vizdoom","gymnasium","numpy"]},
          "python":sys.version,"environment_config":vars(cfg),"summaries":summaries}
(REPORT_DIR / "summary.json").write_text(json.dumps(report,indent=2,default=str))
print("Saved:",REPORT_DIR)


## Compare with a learned-model report

Optionally set a path to an existing ANN or LIF `evaluation.json`. This compares episode returns, not replay results. Exact seed lists and scenario/frame skip must agree. Positive paired differences favor the learned model; they are not evidence about variation across training seeds.

In [ ]:
LEARNED_REPORT = None  # e.g. OUTPUT_ROOT / "results" / "defend_ann_seed0_v1" / "evaluation.json"
print(f"{'Strategy':26s} {'Mean':>9s} {'Std':>9s} {'SE':>9s} {'Decisions':>11s}")
for row in sorted(summaries,key=lambda r:r["mean_return"],reverse=True):
    print(f"{row['strategy']:26s} {row['mean_return']:9.3f} {row['std_return']:9.3f} {row['SE']:9.3f} {row['mean_decisions']:11.2f}")
if LEARNED_REPORT is not None:
    learned = json.loads(Path(LEARNED_REPORT).read_text())
    assert learned["env"] == ENV_NAME
    assert learned["resolved_training_config"]["env_frameskip"] == FRAME_SKIP
    assert [r["seed"] for r in learned["episodes"]] == EVAL_SEEDS, "Use identical episode seed lists"
    learned_rewards = np.array([r["return"] for r in learned["episodes"]])
    print("Learned checkpoint:",learned["checkpoint"])
    for strategy,records in all_records.items():
        delta = learned_rewards - np.array([r["return"] for r in records])
        print(strategy,"mean paired improvement:",delta.mean(),"paired SE:",delta.std()/np.sqrt(len(delta)))


## What this tells us

Spinning and shooting is a plausible strategy in this scenario. If a constant rule
matches the learned policy, reward alone does not establish sophisticated aiming.
If the learned policy beats these baselines on matched seeds, it improves on those
specific rules. Compare multiple training seeds before generalizing to ANN versus LIF.
These validation seeds have been inspected; keep another episode set for final reporting.

We do not teach RL these scripts in this notebook. The learned policy samples actions
and uses reward-driven updates. Forcing a rule during evaluation overrides the policy.
Imitation learning or expert-guided pretraining could teach a script, but would be a
separate training method and should be reported separately. No neural weights change here.
